# Можем ли мы отличить сорняки от рассады?

Приступим к задаче классификации на картинках. Реализуем программу, которая определяет тип рассады на изображении. 

Для того, чтобы определить характерные особенности каждого типа рассады, у нас есть train. Train это папка, в которой картинки уже классифицированы и лежат в соответствующих папках. Исходя из этой информации мы можем найти признаки, присущие конкретному растению.

Проверка решения будет происходить на test. В папке test уже нет метки класса для каждой картинки. 

[Ссылка на Яндекс-диск](https://yadi.sk/d/0Zzp0klXT0iRmA), все картинки тут.

Примеры изображений для теста:
<table><tr>
    <td> <img src="https://i.ibb.co/tbqR37m/fhj.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/6yL3Wmt/sfg.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/pvn7NvF/asd.png" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

## Описание алгоритма

1. Выделим на каждом изображении конкретное растение - выделим на изображении оттенки зеленого
2. Для каждого растения train выборки извлечем дескрипторы с помощью метода SIFT
3. Обучаем кластеризатор KMeans на дескрипторах train-выборки
4. Для каждого изображения из train-выборки строим гистограмму - распределение дескрипторов с изображения по кластерам
5. Полученные гистограммы используем для обучения классификатора SVC, который будет предсказывать класс изображения по распределению дескрипторов с этого изображения
6. Проверяем алгоритм на тестовой выборке
7. (Спойлер) Радуемся хорошему результату!

### Алгоритм SIFT (Scale-Invariant Feature Transform) — метод выделения ключевых точек и их дескрипторов, устойчивых к масштабу, повороту и освещению.

### Основные этапы:
#### 1) Построение масштабного пространства

Изображение преобразуется в разные масштабы с помощью гауссова размытия (Gaussian Blur) и пирамиды масштабов (октавы).

#### 2) Поиск экстремумов в пространстве масштабов

Находит ключевые точки как локальные экстремумы в разностях гауссовых изображений (DoG — Difference of Gaussians).

#### 3) Уточнение положения ключевых точек

Отсеиваются неустойчивые точки (с низкой контрастностью или лежащие на границах).

#### 4) Ориентация ключевых точек

Для каждой точки вычисляется направление градиента в окрестности, чтобы обеспечить инвариантность к повороту.

#### 5) Построение дескриптора

Вокруг ключевой точки анализируются градиенты в 16×16 областях, разбитых на 4×4 подокна.

Формируется гистограмма из 8 направлений для каждого подокна → итого 128-мерный дескриптор (4×4×8).

### Дескрипторы SIFT устойчивы к масштабу, повороту, частично — к освещению и affine-искажениям.

In [50]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC

### Вспомогательные функции:

In [51]:
def extract_plants(image):
    """
    Функция для выделения растений и удаления фона.
    Заменяет фон на черный, оставляя только области с оттенками зеленого.
    """
    
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Диапазон для зелёного цвета 
    lower_green = np.array([35, 40, 40])  
    upper_green = np.array([85, 255, 255])  

    mask = cv2.inRange(hsv, lower_green, upper_green)

    result = cv2.bitwise_and(image, image, mask=mask)

    return result

def process_images_to_arrays(input_folder):
    """
    Функция для обработки изображений из папки и сохранения в массивы
    """
    images = []   
    labels = []
    for class_name in os.listdir(input_folder):
        class_input_path = os.path.join(input_folder, class_name).replace('\\', '/')

        if os.path.isdir(class_input_path):
            for image_name in os.listdir(class_input_path):
                image_path = os.path.join(class_input_path, image_name).replace('\\', '/')

                image = cv2.imread(image_path)
                processed_image = extract_plants(image)
                images.append(processed_image) 
                labels.append(class_name)  

    return images, labels

def extract_sift_features(images):
    """
    Извлечение признаков с изображения с помощью метода SIFT
    """
    sift = cv2.SIFT_create()
    descriptors_list = []  
    for img in images:
        keypoints, descriptors = sift.detectAndCompute(img, None)
        descriptors_list.append(descriptors)
    return descriptors_list
 
def cluster_descriptors(descriptors_list, n_clusters=100):
    """
    Функция для кластеризация дескрипторов
    """
    all_descriptors = np.vstack(descriptors_list)
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans.fit(all_descriptors)
    return kmeans

def create_bovw_histogram(descriptors_list, kmeans, n_clusters):
    """
    Функция для построения распределения дескрипторов по кластерам
    """
    histograms = []
    for descriptors in descriptors_list:
        histogram = np.zeros(n_clusters)
        if descriptors is not None:
            clusters = kmeans.predict(descriptors)
            for cluster in clusters:
                histogram[cluster] += 1
        histograms.append(histogram)
    return np.array(histograms)


### Формируем тренировочную выборку:

In [52]:
train_path = '/plants/train'

train_images, train_labels = process_images_to_arrays(train_path)

# Извлекаем дескрипторы с тренировочной выборки
train_descriptors = extract_sift_features(train_images)

# Кодирование меток классов
label_encoder = LabelEncoder()
train_labels_encoded = label_encoder.fit_transform(train_labels)

# Кластеризация
n_clusters = 100
kmeans = cluster_descriptors(train_descriptors, n_clusters)
train_histograms = create_bovw_histogram(train_descriptors, kmeans, n_clusters)

In [43]:
print("Метки в порядке кодирования:", label_encoder.classes_)

Метки в порядке кодирования: ['Loose_Silky-bent' 'Maize' 'Scentless_Mayweed'
 'Small-flowered_Cranesbill']


### Делим данные и обучаем модель SVC:

In [53]:
X_train, X_val, y_train, y_val = train_test_split(train_histograms, train_labels_encoded, test_size=0.2, random_state=42)

svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train, y_train)


SVC(kernel='linear', probability=True, random_state=42)

### Оцениваем точность на валидационной выборке:

In [44]:
y_val_pred = svm_model.predict(X_val)
accuracy = accuracy_score(y_val, y_val_pred)
print(f"Validation Accuracy: {accuracy:.2f}")

test_path = '/plants/test'  


Validation Accuracy: 1.00


#### Качество на валидации: 100%

### Формируем тестовую выборку:

In [45]:
test_images = []
processed_test_images = []

for image_name in os.listdir(test_path):
    image_path = os.path.join(test_path, image_name).replace('\\', '/')
    image = cv2.imread(image_path)
    processed_image = extract_plants(image)
    test_images.append(image)
    processed_test_images.append(processed_image) 

### Получаем предсказание на тестовой выборке:

In [46]:
# Извлечение дескрипторов 
test_descriptors = extract_sift_features(processed_test_images)
test_histograms = create_bovw_histogram(test_descriptors, kmeans, n_clusters)

# Прогнозирование
predicted_classes = svm_model.predict(test_histograms)
predicted_labels = label_encoder.inverse_transform(predicted_classes)

print(predicted_classes)

[3 1 0 1 1 0 3 2 3 2 3 0 3 2 2 3 1 3 2 2 1 3 3 0 2 0 0 1 2 3 2 1 1]


### Истинные метки для тестовой выборки:

In [47]:
true_labels = [3, 1, 0, 1, 1, 0, 3, 2, 3, 2, 3, 0, 3, 2, 2, 3, 1, 3, 2, 2, 1, 3, 3, 0, 2, 0, 0, 1, 2, 3, 2, 1, 1]

### Смотрим на результат:

In [48]:
sum = 0
for i in range(len(true_labels)):
    if predicted_classes[i] != true_labels[i]:
        sum += 1

print('mistakes_count = ', sum )

mistakes_count =  0


### Все предсказания верные!

### Вывод и результаты: Нам удалось сформировать алгоритм, позволяющий со 100% точностью классифицировать растения на изображениях. 

### Мы убедились в том, что использование особых точек или дескрипторов изображения, извлеченных с помощью метода SIFT, в качестве признаков для классификатора дает отличные результаты. 